# 0826_dongjin_026_saved_ensemble_shap

This notebook performs SHAP and threshold extraction natively on the **saved** `0825_peace_005_type_expert_fold_ensemble.pkl` model bundle.
By using the saved `.pkl` file, we verify that serialization works and we avoid redundant retraining.

For each inspection type:
1. We load the 4 checkpoint models directly from the `.pkl`.
2. We compute ensemble probability on the final Test Set and identify False Positives (False Calls).
3. We compute SHAP values and extract decision boundaries.


In [1]:
import gc
import json
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import shap
import xgboost
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Configuration
DATA_PATH = Path("../data/raw/dataset.csv")
MAPPING_PATH = Path("../data/raw/mapping.json")
MODEL_BUNDLE_PATH = Path("../models/0825_peace_005_type_expert_fold_ensemble.pkl")
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5


E:\pro_newton\제조 AI\팀 과제\siemens_aoi_ML_practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Data, Mapping, and Model Bundle

In [2]:
# Load Model Bundle
with open(MODEL_BUNDLE_PATH, 'rb') as f:
    bundle = pickle.load(f)

inspection_types = bundle['inspection_types']
feature_columns_by_type = bundle['feature_columns_by_type']
checkpoints = bundle['ensemble_checkpoints']

# Load Data
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
if raw_df.columns[0].startswith("Unnamed:") or raw_df.columns[0] == "":
    raw_df = raw_df.rename(columns={raw_df.columns[0]: RECORD_ID})

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)

meta_columns = [col for col in raw_df.columns if col.startswith("meta_feat")]

def make_preprocessor(feature_columns):
    categorical = [c for c in meta_columns if c in feature_columns]
    continuous = [c for c in feature_columns if c not in categorical]
    return ColumnTransformer(
        transformers=[
            ("categorical", OneHotEncoder(handle_unknown="ignore", dtype=np.float32), categorical),
            ("continuous", "passthrough", continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )

print("Model and Data loaded. Types:", inspection_types)


Model and Data loaded. Types: [0, 1, 2, 3, 4]


## 2. Test Set Split

In [3]:
timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index

def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]

validation_end_time = boundary_at(0.80)
test_mask = raw_df[TIME_COLUMN] > validation_end_time
test_df = raw_df.loc[test_mask].copy()

print(f"Test Set Size: {len(test_df)}")


Test Set Size: 88052


## 3. Extract SHAP per Type (Using Saved Models)

In [4]:
final_summary_rows = []

for inspection_type in inspection_types:
    print("=" * 80)
    print(f"INSPECTION TYPE: {inspection_type}")
    print("=" * 80)
    
    feature_columns = feature_columns_by_type[inspection_type]
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    y_test_type = type_test[TARGET].astype("int8")
    
    if len(type_test) == 0:
        continue
        
    ensemble_preds = []
    
    for ckpt in checkpoints:
        model_info = bundle['members'][ckpt][inspection_type]
        model = model_info['model']
        preprocessor = model_info['preprocessor']
        X_test_encoded = preprocessor.transform(type_test[feature_columns])
        preds = model.predict_proba(X_test_encoded)[:, 1]
        ensemble_preds.append(preds)
        
    # Find False Calls
    mean_preds = np.mean(np.vstack(ensemble_preds), axis=0)
    test_predictions = (mean_preds >= DECISION_THRESHOLD).astype("int8")
    
    mask_fp = (y_test_type == 0) & (test_predictions == 1)
    X_fp_type_raw = type_test[feature_columns][mask_fp]
    
    print(f"\n  Total False Calls: {len(X_fp_type_raw)}\n")
    if len(X_fp_type_raw) == 0:
        continue
        
    # SHAP Analysis
    feature_abs_shap_sum = {}
    all_trees = []
    
    for ckpt in checkpoints:
        model_info = bundle['members'][ckpt][inspection_type]
        model = model_info['model']
        preprocessor = model_info['preprocessor']
        encoded_feature_names = preprocessor.get_feature_names_out()
        
        X_fp_type_encoded = preprocessor.transform(X_fp_type_raw)
        if hasattr(X_fp_type_encoded, "toarray"):
            X_fp_type_encoded = X_fp_type_encoded.toarray()
            
        explainer = shap.TreeExplainer(model)
        shap_values_fp = explainer.shap_values(X_fp_type_encoded)
        
        # mean absolute SHAP for this checkpoint across False Calls
        mean_abs_shap_ckpt = np.abs(shap_values_fp).mean(axis=0)
        
        for name, val in zip(encoded_feature_names, mean_abs_shap_ckpt):
            feature_abs_shap_sum[name] = feature_abs_shap_sum.get(name, 0.0) + val
            
        # Thresholds
        trees_df = model.get_booster().trees_to_dataframe()
        feature_map = {f"f{i}": name for i, name in enumerate(encoded_feature_names)}
        trees_df['FeatureName'] = trees_df['Feature'].map(feature_map)
        all_trees.append(trees_df)
        
    # Average across the 4 checkpoints
    for name in feature_abs_shap_sum:
        feature_abs_shap_sum[name] /= len(checkpoints)
        
    # Sort and get top 10
    top_features = sorted(feature_abs_shap_sum.keys(), key=lambda k: feature_abs_shap_sum[k], reverse=True)[:10]
    
    print("  Top Features Driving False Calls (according to Mean Ensemble SHAP):")
    top_summary = []
    for i, feature in enumerate(top_features, 1):
        shap_val = feature_abs_shap_sum[feature]
        top_summary.append(f"{feature} ({shap_val:.4f})")
        print(f"    {i}. {feature} (Mean |SHAP|: {shap_val:.4f})")
        
    final_summary_rows.append({
        "Inspection Type": inspection_type,
        "False Calls": len(X_fp_type_raw),
        "Top Features (|SHAP|)": " <br> ".join([f"{i}. {val}" for i, val in enumerate(top_summary, 1)])
    })
    
    # Thresholds
    print("\n  -- Native Thresholds for Top Features (Aggregated) --")
    merged_trees = pd.concat(all_trees, ignore_index=True)
    
    for feature in top_features:
        feature_nodes = merged_trees[merged_trees['FeatureName'] == feature]
        if len(feature_nodes) == 0:
            continue
            
        threshold_agg = feature_nodes.groupby('Split').agg(
            Frequency=('Split', 'count'),
            Total_Cover=('Cover', 'sum')
        ).sort_values(by='Total_Cover', ascending=False)
        
        top_5 = threshold_agg.head(3)
        thresholds_str = ", ".join([f"< {th:.4f} (Cover: {row['Total_Cover']:.0f})" for th, row in top_5.iterrows()])
        print(f"    * {feature}: {thresholds_str}")
        
    print("\n")


INSPECTION TYPE: 0



  Total False Calls: 0

INSPECTION TYPE: 1



  Total False Calls: 71



  Top Features Driving False Calls (according to Mean Ensemble SHAP):
    1. continuous__inspection_feat24 (Mean |SHAP|: 1.6863)
    2. continuous__inspection_feat48 (Mean |SHAP|: 1.0508)
    3. continuous__inspection_feat8 (Mean |SHAP|: 0.7661)
    4. categorical__meta_feat4_28 (Mean |SHAP|: 0.7565)
    5. categorical__meta_feat1_22 (Mean |SHAP|: 0.4671)
    6. continuous__inspection_feat1 (Mean |SHAP|: 0.3323)
    7. categorical__meta_feat4_1 (Mean |SHAP|: 0.2901)
    8. continuous__inspection_feat25 (Mean |SHAP|: 0.2525)
    9. continuous__inspection_feat2 (Mean |SHAP|: 0.2282)
    10. continuous__inspection_feat4 (Mean |SHAP|: 0.2212)

  -- Native Thresholds for Top Features (Aggregated) --
    * continuous__inspection_feat24: < 0.0333 (Cover: 13019), < 0.6125 (Cover: 5379), < 0.2167 (Cover: 4896)
    * continuous__inspection_feat48: < 0.0708 (Cover: 11634), < 0.1833 (Cover: 6707), < 0.1167 (Cover: 6693)
    * continuous__inspection_feat8: < 0.6297 (Cover: 3616), < 0.7378 (Cover: 3

    * continuous__inspection_feat2: < 0.6644 (Cover: 9119), < 0.7013 (Cover: 7572), < 0.6275 (Cover: 4741)
    * continuous__inspection_feat4: < 0.4346 (Cover: 15458), < 0.4215 (Cover: 10535), < 0.4280 (Cover: 7986)


INSPECTION TYPE: 2



  Total False Calls: 3



  Top Features Driving False Calls (according to Mean Ensemble SHAP):
    1. continuous__inspection_feat96 (Mean |SHAP|: 1.2978)
    2. categorical__meta_feat1_27 (Mean |SHAP|: 1.2666)
    3. categorical__meta_feat4_3 (Mean |SHAP|: 0.6229)
    4. continuous__inspection_feat22 (Mean |SHAP|: 0.5349)
    5. continuous__inspection_feat95 (Mean |SHAP|: 0.5153)
    6. categorical__meta_feat4_41 (Mean |SHAP|: 0.4795)
    7. continuous__inspection_feat28 (Mean |SHAP|: 0.4008)
    8. continuous__inspection_feat12 (Mean |SHAP|: 0.3461)
    9. continuous__inspection_feat1 (Mean |SHAP|: 0.3436)
    10. categorical__meta_feat1_2 (Mean |SHAP|: 0.3053)

  -- Native Thresholds for Top Features (Aggregated) --
    * continuous__inspection_feat96: < 0.2653 (Cover: 27467), < 0.3163 (Cover: 19293), < 0.3551 (Cover: 17733)
    * categorical__meta_feat1_27: < 2.0000 (Cover: 44604)
    * categorical__meta_feat4_3: < 2.0000 (Cover: 14698)
    * continuous__inspection_feat22: < 0.7593 (Cover: 10548), < 0.8465 


  Total False Calls: 92



  Top Features Driving False Calls (according to Mean Ensemble SHAP):
    1. continuous__inspection_feat95 (Mean |SHAP|: 0.9781)
    2. continuous__inspection_feat22 (Mean |SHAP|: 0.8618)
    3. categorical__meta_feat4_7 (Mean |SHAP|: 0.8563)
    4. continuous__inspection_feat12 (Mean |SHAP|: 0.8067)
    5. categorical__meta_feat1_27 (Mean |SHAP|: 0.6507)
    6. continuous__inspection_feat96 (Mean |SHAP|: 0.5000)
    7. categorical__meta_feat4_41 (Mean |SHAP|: 0.3480)
    8. categorical__meta_feat1_24 (Mean |SHAP|: 0.2892)
    9. continuous__inspection_feat4 (Mean |SHAP|: 0.2873)
    10. categorical__meta_feat2_2 (Mean |SHAP|: 0.2332)

  -- Native Thresholds for Top Features (Aggregated) --
    * continuous__inspection_feat95: < 0.1538 (Cover: 38332), < 0.3077 (Cover: 29674), < 0.1615 (Cover: 15664)
    * continuous__inspection_feat22: < 0.9627 (Cover: 12650), < 0.7676 (Cover: 11180), < 0.8672 (Cover: 7223)
    * categorical__meta_feat4_7: < 2.0000 (Cover: 48556)
    * continuous__insp


  Total False Calls: 0



## 4. Final Summary Table

In [5]:
import IPython.display as display
summary_df = pd.DataFrame(final_summary_rows)
display.display(display.HTML(summary_df.to_html(escape=False, index=False)))

with open("../docs/experiments/0826_dongjin_026_saved_ensemble_shap.md", "w", encoding="utf-8") as f:
    f.write("# 0826_dongjin_026_saved_ensemble_shap\n\n")
    f.write("## Overview\n")
    f.write("Extracted False Calls and SHAP values natively using the saved `models/0825_peace_005_type_expert_fold_ensemble.pkl` bundle.\n")
    f.write("## Top 10 Features Driving False Calls by Type\n\n")
    f.write(summary_df.to_markdown(index=False))
    f.write("\n")


Inspection Type,False Calls,Top Features (|SHAP|)
1,71,1. continuous__inspection_feat24 (1.6863) 2. continuous__inspection_feat48 (1.0508) 3. continuous__inspection_feat8 (0.7661) 4. categorical__meta_feat4_28 (0.7565) 5. categorical__meta_feat1_22 (0.4671) 6. continuous__inspection_feat1 (0.3323) 7. categorical__meta_feat4_1 (0.2901) 8. continuous__inspection_feat25 (0.2525) 9. continuous__inspection_feat2 (0.2282) 10. continuous__inspection_feat4 (0.2212)
2,3,1. continuous__inspection_feat96 (1.2978) 2. categorical__meta_feat1_27 (1.2666) 3. categorical__meta_feat4_3 (0.6229) 4. continuous__inspection_feat22 (0.5349) 5. continuous__inspection_feat95 (0.5153) 6. categorical__meta_feat4_41 (0.4795) 7. continuous__inspection_feat28 (0.4008) 8. continuous__inspection_feat12 (0.3461) 9. continuous__inspection_feat1 (0.3436) 10. categorical__meta_feat1_2 (0.3053)
3,92,1. continuous__inspection_feat95 (0.9781) 2. continuous__inspection_feat22 (0.8618) 3. categorical__meta_feat4_7 (0.8563) 4. continuous__inspection_feat12 (0.8067) 5. categorical__meta_feat1_27 (0.6507) 6. continuous__inspection_feat96 (0.5000) 7. categorical__meta_feat4_41 (0.3480) 8. categorical__meta_feat1_24 (0.2892) 9. continuous__inspection_feat4 (0.2873) 10. categorical__meta_feat2_2 (0.2332)
